In [1]:
# | Goal                                  | Meaning                   |
# | ------------------------------------- | ------------------------- |
# | train ML model                        | predict default risk      |
# | validate engineered features          | see what actually matters |
# | explain feature importance            | enterprise explainability |
# | evaluate predictive power             | ROC-AUC, classification   |
# | identify strongest behavioral signals | business intelligence     |


In [2]:
# IMPORT LIBRARIES ⭐⭐⭐



import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (

    classification_report,

    roc_auc_score,

    confusion_matrix
)

In [3]:
# ============================================================
# LOAD ENGINEERED DATASET
# ============================================================

application_train = pd.read_csv(

    "../data/processed/engineered_credit_risk_dataset.csv"
)

application_train.shape

(307511, 145)

In [4]:
# SELECT FEATURES ⭐⭐⭐

# Now we build:

# modeling dataset.



feature_columns = [

    "ANNUITY_TO_INCOME",

    "CREDIT_TO_INCOME",

    "FREE_CASH_FLOW",

    "DEBT_TO_INCOME",

    "DEBT_PER_BUREAU_RECORD",

    "APPLICATIONS_PER_INCOME",

    "LATE_PAYMENT_COUNT",

    "MISSED_PAYMENTS_PER_LOAN",

    "AVG_PAYMENT_DELAY",

    "RECENT_LATE_PAYMENT_COUNT",

    "REPAYMENT_STABILITY",

    "YEARS_EMPLOYED",

    "EMPLOYMENT_TO_AGE_RATIO",

    "BEHAVIORAL_RISK_SCORE"
]

In [5]:
# BUILD X AND y ⭐⭐⭐
X = application_train[
    feature_columns
]

y = application_train[
    "TARGET"
]

In [6]:
# HANDLE MISSING VALUES ⭐⭐⭐

X = X.fillna(0)

In [7]:
# TRAIN / TEST SPLIT ⭐⭐⭐

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [8]:
# IMPORT SMOTE ⭐⭐⭐

from imblearn.over_sampling import SMOTE

In [9]:
# APPLY SMOTE ONLY TO TRAINING DATA ⭐⭐⭐
smote = SMOTE(

    random_state=42
)

X_train_resampled, y_train_resampled = (

    smote.fit_resample(

        X_train,
        y_train
    )
)

In [11]:
# VALIDATE BALANCE ⭐⭐⭐


print(

    y_train.value_counts()
)

print(
    "----------------------"
)

print(

    y_train_resampled.value_counts()
)

TARGET
0    226148
1     19860
Name: count, dtype: int64
----------------------
TARGET
0    226148
1    226148
Name: count, dtype: int64


In [15]:
# MPORT STANDARD SCALER ⭐⭐⭐
from sklearn.preprocessing import StandardScaler
# SCALE TRAIN + TEST ⭐⭐⭐

# Run BEFORE model training:

# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(

    X_train_resampled
)

X_test_scaled = scaler.transform(

    X_test
)

In [12]:
# TRAIN LOGISTIC REGRESSION ⭐⭐⭐

# ============================================================
# LOGISTIC REGRESSION MODEL
# ============================================================
# WHY LOGISTIC REGRESSION FIRST ⭐⭐⭐

# Because:
# coefficients are:

# explainable.

# This is VERY important in banking.

# Unlike black-box models,
# you can directly inspect:

# feature direction,
# feature impact,
# risk drivers.

# THIS matters enormously in:

# underwriting,
# compliance,
# model governance,
# regulatory reviews.

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(

    max_iter=1000,

    random_state=42
)

model.fit(

    X_train_resampled,

    y_train_resampled
)

print(
    "Model training completed."
)

Model training completed.


/home/muzzi/first_project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
# GENERATE PREDICTIONS ⭐⭐⭐

# After training:

# ============================================================
# MODEL PREDICTIONS
# ============================================================

predictions = model.predict(

    X_test
)

probabilities = model.predict_proba(

    X_test
)[:,1]

print(
    "Predictions generated successfully."
)

Predictions generated successfully.


In [14]:
# CLASSIFICATION REPORT ⭐⭐⭐

# Now evaluate model performance.

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print(

    classification_report(

        y_test,

        predictions
    )
)

              precision    recall  f1-score   support

           0       0.93      0.51      0.66     56538
           1       0.09      0.54      0.15      4965

    accuracy                           0.51     61503
   macro avg       0.51      0.52      0.41     61503
weighted avg       0.86      0.51      0.62     61503



In [16]:
# RETRAIN MODEL ⭐⭐⭐
# Now retrain using:
# scaled data.

# ============================================================
# LOGISTIC REGRESSION MODEL
# ============================================================

model = LogisticRegression(

    max_iter=3000,

    random_state=42
)

model.fit(

    X_train_scaled,

    y_train_resampled
)

print(
    "Model training completed."
)

Model training completed.


In [17]:
# PREDICT USING SCALED TEST SET ⭐⭐⭐

# VERY important.

# ============================================================
# MODEL PREDICTIONS
# ============================================================

predictions = model.predict(

    X_test_scaled
)

probabilities = model.predict_proba(

    X_test_scaled
)[:,1]

In [18]:
# CLASSIFICATION REPORT ⭐⭐⭐

# Now evaluate model performance.

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print(

    classification_report(

        y_test,

        predictions
    )
)

              precision    recall  f1-score   support

           0       0.93      0.60      0.73     56538
           1       0.09      0.47      0.16      4965

    accuracy                           0.59     61503
   macro avg       0.51      0.54      0.44     61503
weighted avg       0.86      0.59      0.69     61503



In [19]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    probabilities
)

print(
    f"ROC AUC Score: {auc:.4f}"
)

ROC AUC Score: 0.5565


In [21]:
## XGboost model

# MPORT LIBRARIES ⭐⭐⭐


# ============================================================
# XGBOOST + SHAP IMPORTS
# ============================================================

from xgboost import XGBClassifier

import shap


In [22]:
# TRAIN XGBOOST MODEL ⭐⭐⭐

# ============================================================
# XGBOOST MODEL
# ============================================================

xgb_model = XGBClassifier(

    n_estimators=200,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

xgb_model.fit(

    X_train_resampled,

    y_train_resampled
)

print(
    "XGBoost training completed."
)

XGBoost training completed.


In [23]:
# GENERATE PREDICTIONS ⭐⭐⭐
# ============================================================
# XGBOOST PREDICTIONS
# ============================================================

xgb_predictions = xgb_model.predict(

    X_test
)

xgb_probabilities = xgb_model.predict_proba(

    X_test
)[:,1]

print(
    "Predictions generated successfully."
)


Predictions generated successfully.


In [24]:
# STEP 5 — EVALUATE MODEL ⭐⭐⭐
# CLASSIFICATION REPORT ⭐⭐⭐
# ============================================================
# XGBOOST CLASSIFICATION REPORT
# ============================================================

print(

    classification_report(

        y_test,

        xgb_predictions
    )
)


              precision    recall  f1-score   support

           0       0.92      0.99      0.95     56538
           1       0.15      0.02      0.03      4965

    accuracy                           0.91     61503
   macro avg       0.54      0.50      0.49     61503
weighted avg       0.86      0.91      0.88     61503



In [25]:
# ROC-AUC ⭐⭐⭐ MOST IMPORTANT
# ============================================================
# XGBOOST ROC-AUC
# ============================================================

xgb_auc = roc_auc_score(

    y_test,

    xgb_probabilities
)

print(
    f"XGBoost ROC AUC: {xgb_auc:.4f}"
)

XGBoost ROC AUC: 0.5916


In [39]:
# ============================================================
# LOAD UPDATED ENGINEERED DATASET
# ============================================================

application_train = pd.read_csv(

    "../data/processed/engineered_credit_risk_dataset.csv"
)

application_train.shape

(307511, 169)

In [40]:

# ============================================================
# FEATURE SELECTION
# ============================================================
# ============================================================
# FINAL ENTERPRISE FEATURE SET ⭐⭐⭐
# ============================================================
# ============================================================
# FINAL ENTERPRISE FEATURE SET ⭐⭐⭐
# ============================================================

feature_columns = [

    # ========================================================
    # AFFORDABILITY FEATURES
    # ========================================================

    "ANNUITY_TO_INCOME",

    "CREDIT_TO_INCOME",

    "FREE_CASH_FLOW",

    "DEBT_TO_INCOME",

    "CREDIT_TO_ANNUITY_RATIO",

    "CREDIT_TO_GOODS_RATIO",

    "DOWN_PAYMENT",


    # ========================================================
    # EXTERNAL RISK / EXT_SOURCE FEATURES ⭐⭐⭐
    # ========================================================

    "EXT_SOURCE_1",

    "EXT_SOURCE_2",

    "EXT_SOURCE_3",

    "EXT_SOURCE_MEAN",

    "EXT_SOURCE_STD",

    "CREDIT_EXT_RATIO",


    # ========================================================
    # LEVERAGE + EXPOSURE FEATURES
    # ========================================================

    "DEBT_PER_BUREAU_RECORD",

    "OVERDUE_PER_BUREAU_RECORD",

    "ACTIVE_DEBT_RATIO",

    "MEAN_DAYS_CREDIT",

    "LAST_ACTIVE_DAYS_CREDIT",


    # ========================================================
    # BORROWING BEHAVIOR FEATURES
    # ========================================================

    "APPLICATIONS_PER_INCOME",

    "PREVIOUS_APPLICATION_COUNT",

    "RECENT_APPLICATION_COUNT",

    "BORROWING_ACCELERATION_RATIO",


    # ========================================================
    # REPAYMENT BEHAVIOR FEATURES
    # ========================================================

    "LATE_PAYMENT_COUNT",

    "MISSED_PAYMENTS_PER_LOAN",

    "AVG_PAYMENT_DELAY",

    "AVG_PAYMENT_DEFICIT",

    "REPAYMENT_STABILITY",


    # ========================================================
    # TEMPORAL REPAYMENT FEATURES
    # ========================================================

    "LATE_PAYMENTS_LAST_90D",

    "AVG_PAYMENT_DELAY_LAST_90D",

    "RECENT_TO_HISTORICAL_DELAY_RATIO",

    "RECENT_DELAY_TREND",

    "RECENT_PAYMENT_DEFICIT",

    "PAYMENT_DEFICIT_TREND",

    "RECENT_PAYMENT_STABILITY",


    # ========================================================
    # EMPLOYMENT + DEMOGRAPHIC FEATURES
    # ========================================================

    "YEARS_EMPLOYED",

    "EMPLOYMENT_TO_AGE_RATIO",

    "AGE_YEARS",


    # ========================================================
    # HISTORICAL REPAYMENT CAPACITY
    # ========================================================

    "MAX_INSTALLMENT",

    "ANNUITY_TO_MAX_INSTALLMENT_RATIO",


    # ========================================================
    # COMPOSITE RISK FEATURES
    # ========================================================

    "BEHAVIORAL_RISK_SCORE"
]
# ============================================================
# BUILD MODELING DATASET
# ============================================================

X = application_train[
    feature_columns
]

y = application_train[
    "TARGET"
]

# ------------------------------------------------------------
# HANDLE NULLS
# ------------------------------------------------------------

X = X.fillna(0)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print(y_train.value_counts())


# ============================================================
# HANDLE CLASS IMBALANCE — SMOTE
# ============================================================

smote = SMOTE(

    random_state=42
)

X_train_resampled, y_train_resampled = (

    smote.fit_resample(

        X_train,
        y_train
    )
)

print("----------------------------")

print(y_train_resampled.value_counts())



TARGET
0    226148
1     19860
Name: count, dtype: int64
----------------------------
TARGET
0    226148
1    226148
Name: count, dtype: int64


In [41]:

# ============================================================
# XGBOOST MODEL
# ============================================================

xgb_model = XGBClassifier(

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

xgb_model.fit(

    X_train_resampled,

    y_train_resampled
)

print(
    "XGBoost training completed."
)


# ============================================================
# PREDICTIONS
# ============================================================

xgb_predictions = xgb_model.predict(

    X_test
)

xgb_probabilities = xgb_model.predict_proba(

    X_test
)[:,1]

print(
    "Predictions generated successfully."
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    "\nCLASSIFICATION REPORT\n"
)

print(

    classification_report(

        y_test,

        xgb_predictions
    )
)


# ============================================================
# ROC AUC SCORE
# ============================================================

xgb_auc = roc_auc_score(

    y_test,

    xgb_probabilities
)

print(
    f"\nXGBoost ROC AUC: {xgb_auc:.4f}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    xgb_predictions
)

print(
    "\nCONFUSION MATRIX\n"
)

print(cm)

XGBoost training completed.
Predictions generated successfully.

CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.41      0.05      0.09      4965

    accuracy                           0.92     61503
   macro avg       0.67      0.52      0.52     61503
weighted avg       0.88      0.92      0.89     61503


XGBoost ROC AUC: 0.7492

CONFUSION MATRIX

[[56195   343]
 [ 4723   242]]


In [46]:
# ============================================================
# CUSTOM THRESHOLD PREDICTIONS
# ============================================================

custom_predictions = (

    xgb_probabilities > 0.20
).astype(int)

print(

    classification_report(

        y_test,

        custom_predictions
    )
)


# ============================================================
# ROC AUC SCORE
# ============================================================

xgb_auc = roc_auc_score(

    y_test,

    xgb_probabilities
)

print(
    f"\nXGBoost ROC AUC: {xgb_auc:.4f}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    custom_predictions
)

print(cm)

              precision    recall  f1-score   support

           0       0.95      0.87      0.90     56538
           1       0.22      0.43      0.29      4965

    accuracy                           0.83     61503
   macro avg       0.58      0.65      0.60     61503
weighted avg       0.89      0.83      0.85     61503


XGBoost ROC AUC: 0.7492
[[48999  7539]
 [ 2837  2128]]


In [47]:
# ============================================================
# XGBOOST ENTERPRISE OPTIMIZATION ⭐⭐⭐
# OPTIMIZE FOR:
# - ROC AUC
# - RECALL
# - PRECISION
# - F1 SCORE
#
# WITH:
# custom operational threshold
# ============================================================

from sklearn.model_selection import ParameterGrid

from sklearn.metrics import (

    roc_auc_score,

    precision_score,

    recall_score,

    f1_score
)

# ============================================================
# PARAMETER GRID
# ============================================================

param_grid = {

    "max_depth": [4, 6, 8],

    "learning_rate": [0.03, 0.05, 0.1],

    "n_estimators": [200, 300],

    "subsample": [0.8],

    "colsample_bytree": [0.8]
}

# ============================================================
# THRESHOLDS TO TEST ⭐⭐⭐
# ============================================================

thresholds = [

    0.10,

    0.15,

    0.20,

    0.25,

    0.30
]

# ============================================================
# STORE RESULTS
# ============================================================

results = []

# ============================================================
# HYPERPARAMETER + THRESHOLD SEARCH
# ============================================================

for params in ParameterGrid(param_grid):

    print(f"\nTesting Parameters: {params}")

    # --------------------------------------------------------
    # TRAIN MODEL
    # --------------------------------------------------------

    model = XGBClassifier(

        random_state=42,

        eval_metric="logloss",

        **params
    )

    model.fit(

        X_train_resampled,

        y_train_resampled
    )

    # --------------------------------------------------------
    # PREDICT PROBABILITIES
    # --------------------------------------------------------

    probabilities = model.predict_proba(
        X_test
    )[:,1]

    # --------------------------------------------------------
    # ROC AUC
    # --------------------------------------------------------

    auc = roc_auc_score(

        y_test,

        probabilities
    )

    # --------------------------------------------------------
    # TEST DIFFERENT THRESHOLDS
    # --------------------------------------------------------

    for threshold in thresholds:

        predictions = (

            probabilities > threshold
        ).astype(int)

        precision = precision_score(

            y_test,

            predictions,

            zero_division=0
        )

        recall = recall_score(

            y_test,

            predictions
        )

        f1 = f1_score(

            y_test,

            predictions
        )

        results.append({

            "max_depth": params["max_depth"],

            "learning_rate": params["learning_rate"],

            "n_estimators": params["n_estimators"],

            "threshold": threshold,

            "roc_auc": auc,

            "precision": precision,

            "recall": recall,

            "f1_score": f1
        })

        print(

            f"Threshold={threshold} | "
            f"AUC={auc:.4f} | "
            f"Precision={precision:.4f} | "
            f"Recall={recall:.4f} | "
            f"F1={f1:.4f}"
        )

    print("------------------------------------------")

# ============================================================
# RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

# ============================================================
# SORT BY F1 SCORE ⭐⭐⭐
# ============================================================

results_df = results_df.sort_values(

    by=[

        "f1_score",

        "roc_auc"
    ],

    ascending=False
)

# ============================================================
# BEST RESULTS
# ============================================================

results_df.head(20)


Testing Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}
Threshold=0.1 | AUC=0.7242 | Precision=0.1006 | Recall=0.9311 | F1=0.1815
Threshold=0.15 | AUC=0.7242 | Precision=0.1202 | Recall=0.8330 | F1=0.2101
Threshold=0.2 | AUC=0.7242 | Precision=0.1378 | Recall=0.7253 | F1=0.2315
Threshold=0.25 | AUC=0.7242 | Precision=0.1551 | Recall=0.6242 | F1=0.2485
Threshold=0.3 | AUC=0.7242 | Precision=0.1733 | Recall=0.5331 | F1=0.2615
------------------------------------------

Testing Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.8}
Threshold=0.1 | AUC=0.7286 | Precision=0.1133 | Recall=0.8731 | F1=0.2006
Threshold=0.15 | AUC=0.7286 | Precision=0.1357 | Recall=0.7543 | F1=0.2300
Threshold=0.2 | AUC=0.7286 | Precision=0.1554 | Recall=0.6352 | F1=0.2497
Threshold=0.25 | AUC=0.7286 | Precision=0.1772 | Recall=0.5345 | F1=0.2661
Threshold=0.3 | AUC=0.7286 | Precisio

,max_depth,learning_rate,n_estimators,threshold,roc_auc,precision,recall,f1_score
82,8,0.10,200,0.20,0.753269,0.248774,0.378046,0.300080
77,6,0.10,300,0.20,0.755631,0.244880,0.385297,0.299444
57,8,0.05,300,0.20,0.753811,0.240752,0.394562,0.299038
86,8,0.10,300,0.15,0.752940,0.214618,0.473716,0.295403
87,8,0.10,300,0.20,0.752940,0.251673,0.356093,0.294912
27,8,0.03,300,0.20,0.748506,0.217959,0.450755,0.293836
72,6,0.10,200,0.20,0.752101,0.231556,0.401410,0.293693
76,6,0.10,300,0.15,0.755631,0.206021,0.509970,0.293480
56,8,0.05,300,0.15,0.753811,0.203437,0.524471,0.293161
67,4,0.10,300,0.20,0.748082,0.224776,0.419537,0.292721


In [49]:
# STEP 5 — VIEW BEST RESULTS ⭐⭐⭐
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="f1_score",

    ascending=False
)

results_df.head(10)

,max_depth,learning_rate,n_estimators,threshold,roc_auc,precision,recall,f1_score
82,8,0.10,200,0.20,0.753269,0.248774,0.378046,0.300080
77,6,0.10,300,0.20,0.755631,0.244880,0.385297,0.299444
57,8,0.05,300,0.20,0.753811,0.240752,0.394562,0.299038
86,8,0.10,300,0.15,0.752940,0.214618,0.473716,0.295403
87,8,0.10,300,0.20,0.752940,0.251673,0.356093,0.294912
27,8,0.03,300,0.20,0.748506,0.217959,0.450755,0.293836
72,6,0.10,200,0.20,0.752101,0.231556,0.401410,0.293693
76,6,0.10,300,0.15,0.755631,0.206021,0.509970,0.293480
56,8,0.05,300,0.15,0.753811,0.203437,0.524471,0.293161
67,4,0.10,300,0.20,0.748082,0.224776,0.419537,0.292721


In [50]:
# TRAIN MULTIPLE MODELS ⭐⭐⭐
# ============================================================
# TRAIN ENSEMBLE MODELS
# ============================================================

model_1 = XGBClassifier(

    max_depth=8,

    learning_rate=0.10,

    n_estimators=200,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

model_2 = XGBClassifier(

    max_depth=6,

    learning_rate=0.10,

    n_estimators=300,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

model_3 = XGBClassifier(

    max_depth=8,

    learning_rate=0.05,

    n_estimators=300,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

# ------------------------------------------------------------
# FIT MODELS
# ------------------------------------------------------------

model_1.fit(
    X_train_resampled,
    y_train_resampled
)

model_2.fit(
    X_train_resampled,
    y_train_resampled
)

model_3.fit(
    X_train_resampled,
    y_train_resampled
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [51]:
# GET PROBABILITIES ⭐⭐⭐
# ============================================================
# PREDICT PROBABILITIES
# ============================================================

prob_1 = model_1.predict_proba(
    X_test
)[:,1]

prob_2 = model_2.predict_proba(
    X_test
)[:,1]

prob_3 = model_3.predict_proba(
    X_test
)[:,1]

In [52]:
# ENSEMBLE AVERAGING ⭐⭐⭐
# ============================================================
# ENSEMBLE PROBABILITIES
# ============================================================

ensemble_probabilities = (

    prob_1
    +
    prob_2
    +
    prob_3

) / 3

In [53]:
# ROC-AUC ⭐⭐⭐
# ============================================================
# ENSEMBLE ROC-AUC
# ============================================================

ensemble_auc = roc_auc_score(

    y_test,

    ensemble_probabilities
)

print(
    f"Ensemble ROC-AUC: {ensemble_auc:.4f}"
)

Ensemble ROC-AUC: 0.7572


In [54]:
# THRESHOLD TESTING ⭐⭐⭐
# ============================================================
# THRESHOLD TESTING
# ============================================================

thresholds = [

    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]

for threshold in thresholds:

    ensemble_predictions = (

        ensemble_probabilities > threshold
    ).astype(int)

    print(f"\nThreshold: {threshold}")

    print(

        classification_report(

            y_test,

            ensemble_predictions
        )
    )

    print("--------------------------------")


Threshold: 0.1
              precision    recall  f1-score   support

           0       0.96      0.70      0.81     56538
           1       0.17      0.68      0.27      4965

    accuracy                           0.70     61503
   macro avg       0.56      0.69      0.54     61503
weighted avg       0.90      0.70      0.77     61503

--------------------------------

Threshold: 0.15
              precision    recall  f1-score   support

           0       0.95      0.83      0.88     56538
           1       0.21      0.51      0.30      4965

    accuracy                           0.80     61503
   macro avg       0.58      0.67      0.59     61503
weighted avg       0.89      0.80      0.84     61503

--------------------------------

Threshold: 0.2
              precision    recall  f1-score   support

           0       0.94      0.90      0.92     56538
           1       0.25      0.39      0.30      4965

    accuracy                           0.86     61503
   macro avg 

In [ ]:
# NOW LOOK AT THE TRADEOFF BEAUTIFULLY ⭐⭐⭐
# THRESHOLD = 0.10
# Metric	Interpretation
# Recall 68%	catch MANY defaults
# Precision 17%	many false alarms

# This behaves like:

# aggressive monitoring system.
# THRESHOLD = 0.20 ⭐⭐⭐
# Metric	Interpretation
# Precision 25%	decent alarm quality
# Recall 39%	meaningful detection

# This is probably:

# your best balanced enterprise setting.
# THRESHOLD = 0.30
# Metric	Interpretation
# Precision 33%	cleaner alerts
# Recall 21%	many defaults missed

# This behaves like:

# conservative approval system.

In [55]:
# SAVE THE BEST 3 MODELS ⭐⭐⭐


import joblib
# SAVE MODEL 1 ⭐⭐⭐
joblib.dump(

    model_1,

    "../models/xgb_model_depth8_lr010_200.pkl"
)
# SAVE MODEL 2 ⭐⭐⭐
joblib.dump(

    model_2,

    "../models/xgb_model_depth6_lr010_300.pkl"
)
# SAVE MODEL 3 ⭐⭐⭐
joblib.dump(

    model_3,

    "../models/xgb_model_depth8_lr005_300.pkl"
)

['../models/xgb_model_depth8_lr005_300.pkl']

In [56]:
# SAVE ENSEMBLE CONFIG ⭐⭐⭐

# VERY enterprise practice.

ensemble_config = {

    "models": [

        "xgb_model_depth8_lr010_200.pkl",

        "xgb_model_depth6_lr010_300.pkl",

        "xgb_model_depth8_lr005_300.pkl"
    ],

    "ensemble_method": "probability_average",

    "recommended_threshold": 0.20
}

joblib.dump(

    ensemble_config,

    "../models/ensemble_config.pkl"
)

['../models/ensemble_config.pkl']

In [57]:
## remove Smote and use average weights

# ============================================================
# LOAD UPDATED ENGINEERED DATASET
# ============================================================

application_train = pd.read_csv(

    "../data/processed/engineered_credit_risk_dataset.csv"
)

application_train.shape

(307511, 169)

In [58]:


# ============================================================
# FEATURE COLUMNS ⭐⭐⭐
# ============================================================

feature_columns = [

    # ========================================================
    # AFFORDABILITY FEATURES
    # ========================================================

    "ANNUITY_TO_INCOME",

    "CREDIT_TO_INCOME",

    "FREE_CASH_FLOW",

    "DEBT_TO_INCOME",

    "CREDIT_TO_ANNUITY_RATIO",

    "CREDIT_TO_GOODS_RATIO",

    "DOWN_PAYMENT",


    # ========================================================
    # EXTERNAL RISK / EXT_SOURCE FEATURES
    # ========================================================

    "EXT_SOURCE_1",

    "EXT_SOURCE_2",

    "EXT_SOURCE_3",

    "EXT_SOURCE_MEAN",

    "EXT_SOURCE_STD",

    "CREDIT_EXT_RATIO",


    # ========================================================
    # LEVERAGE + EXPOSURE FEATURES
    # ========================================================

    "DEBT_PER_BUREAU_RECORD",

    "OVERDUE_PER_BUREAU_RECORD",

    "ACTIVE_DEBT_RATIO",

    "MEAN_DAYS_CREDIT",

    "LAST_ACTIVE_DAYS_CREDIT",


    # ========================================================
    # BORROWING BEHAVIOR FEATURES
    # ========================================================

    "APPLICATIONS_PER_INCOME",

    "PREVIOUS_APPLICATION_COUNT",

    "RECENT_APPLICATION_COUNT",

    "BORROWING_ACCELERATION_RATIO",


    # ========================================================
    # REPAYMENT BEHAVIOR FEATURES
    # ========================================================

    "LATE_PAYMENT_COUNT",

    "MISSED_PAYMENTS_PER_LOAN",

    "AVG_PAYMENT_DELAY",

    "AVG_PAYMENT_DEFICIT",

    "REPAYMENT_STABILITY",


    # ========================================================
    # TEMPORAL REPAYMENT FEATURES
    # ========================================================

    "LATE_PAYMENTS_LAST_90D",

    "AVG_PAYMENT_DELAY_LAST_90D",

    "RECENT_TO_HISTORICAL_DELAY_RATIO",

    "RECENT_DELAY_TREND",

    "RECENT_PAYMENT_DEFICIT",

    "PAYMENT_DEFICIT_TREND",

    "RECENT_PAYMENT_STABILITY",


    # ========================================================
    # EMPLOYMENT + DEMOGRAPHIC FEATURES
    # ========================================================

    "YEARS_EMPLOYED",

    "EMPLOYMENT_TO_AGE_RATIO",

    "AGE_YEARS",


    # ========================================================
    # HISTORICAL REPAYMENT CAPACITY
    # ========================================================

    "MAX_INSTALLMENT",

    "ANNUITY_TO_MAX_INSTALLMENT_RATIO",


    # ========================================================
    # COMPOSITE RISK FEATURES
    # ========================================================

    "BEHAVIORAL_RISK_SCORE"
]


# ============================================================
# TARGET VARIABLE
# ============================================================

target_column = "TARGET"


# ============================================================
# CREATE X AND y
# ============================================================

X = application_train[
    feature_columns
]

y = application_train[
    target_column
]


# ============================================================
# HANDLE MISSING VALUES ⭐⭐⭐
# ============================================================

X = X.fillna(0)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("\nTRAIN TARGET DISTRIBUTION\n")

print(
    y_train.value_counts()
)


# ============================================================
# SCALE_POS_WEIGHT ⭐⭐⭐
# ============================================================

negative_cases = (

    y_train == 0
).sum()

positive_cases = (

    y_train == 1
).sum()

scale_pos_weight = (

    negative_cases
    /
    positive_cases
)

print(
    f"\nscale_pos_weight = {scale_pos_weight:.2f}"
)


# ============================================================
# TRAIN XGBOOST MODEL ⭐⭐⭐
# ============================================================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(

    max_depth=6,

    learning_rate=0.10,

    n_estimators=300,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    eval_metric="logloss"
)

xgb_model.fit(

    X_train,

    y_train
)

print(
    "\nXGBoost training completed successfully."
)


# ============================================================
# PREDICT PROBABILITIES
# ============================================================

xgb_probabilities = xgb_model.predict_proba(

    X_test

)[:,1]


# ============================================================
# CUSTOM THRESHOLD ⭐⭐⭐
# ============================================================

threshold = 0.20

xgb_predictions = (

    xgb_probabilities > threshold
).astype(int)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import (

    classification_report,

    roc_auc_score,

    confusion_matrix
)

print(
    "\nCLASSIFICATION REPORT\n"
)

print(

    classification_report(

        y_test,

        xgb_predictions
    )
)


# ============================================================
# ROC AUC SCORE
# ============================================================

xgb_auc = roc_auc_score(

    y_test,

    xgb_probabilities
)

print(
    f"\nXGBoost ROC AUC: {xgb_auc:.4f}"
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    xgb_predictions
)

print(
    "\nCONFUSION MATRIX\n"
)

print(cm)


TRAIN TARGET DISTRIBUTION

TARGET
0    226148
1     19860
Name: count, dtype: int64

scale_pos_weight = 11.39

XGBoost training completed successfully.

CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       0.98      0.31      0.47     56538
           1       0.11      0.93      0.19      4965

    accuracy                           0.36     61503
   macro avg       0.54      0.62      0.33     61503
weighted avg       0.91      0.36      0.45     61503


XGBoost ROC AUC: 0.7617

CONFUSION MATRIX

[[17577 38961]
 [  357  4608]]


In [ ]:
# ============================================================
# ENTERPRISE XGBOOST OPTIMIZATION ⭐⭐⭐
# NO SMOTE VERSION
#
# OPTIMIZES:
# - Hyperparameters
# - Thresholds
# - Precision
# - Recall
# - F1
# - ROC-AUC
#
# USING:
# scale_pos_weight
# ============================================================

import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import (

    train_test_split,

    ParameterGrid
)

from sklearn.metrics import (

    roc_auc_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix
)


# ============================================================
# LOAD DATASET
# ============================================================

application_train = pd.read_csv(

    "../data/processed/engineered_credit_risk_dataset.csv"
)

print(application_train.shape)


# ============================================================
# FEATURE COLUMNS
# ============================================================

feature_columns = [

    "ANNUITY_TO_INCOME",
    "CREDIT_TO_INCOME",
    "FREE_CASH_FLOW",
    "DEBT_TO_INCOME",
    "CREDIT_TO_ANNUITY_RATIO",
    "CREDIT_TO_GOODS_RATIO",
    "DOWN_PAYMENT",

    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_STD",
    "CREDIT_EXT_RATIO",

    "DEBT_PER_BUREAU_RECORD",
    "OVERDUE_PER_BUREAU_RECORD",
    "ACTIVE_DEBT_RATIO",
    "MEAN_DAYS_CREDIT",
    "LAST_ACTIVE_DAYS_CREDIT",

    "APPLICATIONS_PER_INCOME",
    "PREVIOUS_APPLICATION_COUNT",
    "RECENT_APPLICATION_COUNT",
    "BORROWING_ACCELERATION_RATIO",

    "LATE_PAYMENT_COUNT",
    "MISSED_PAYMENTS_PER_LOAN",
    "AVG_PAYMENT_DELAY",
    "AVG_PAYMENT_DEFICIT",
    "REPAYMENT_STABILITY",

    "LATE_PAYMENTS_LAST_90D",
    "AVG_PAYMENT_DELAY_LAST_90D",
    "RECENT_TO_HISTORICAL_DELAY_RATIO",
    "RECENT_DELAY_TREND",
    "RECENT_PAYMENT_DEFICIT",
    "PAYMENT_DEFICIT_TREND",
    "RECENT_PAYMENT_STABILITY",

    "YEARS_EMPLOYED",
    "EMPLOYMENT_TO_AGE_RATIO",
    "AGE_YEARS",

    "MAX_INSTALLMENT",
    "ANNUITY_TO_MAX_INSTALLMENT_RATIO",

    "BEHAVIORAL_RISK_SCORE"
]


# ============================================================
# TARGET
# ============================================================

target_column = "TARGET"

X = application_train[
    feature_columns
].fillna(0)

y = application_train[
    target_column
]


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)


# ============================================================
# SCALE_POS_WEIGHT ⭐⭐⭐
# ============================================================

negative_cases = (

    y_train == 0
).sum()

positive_cases = (

    y_train == 1
).sum()

scale_pos_weight = (

    negative_cases
    /
    positive_cases
)

print(
    f"\nscale_pos_weight = {scale_pos_weight:.2f}"
)


# ============================================================
# HYPERPARAMETER GRID ⭐⭐⭐
# ============================================================

param_grid = {

    "max_depth": [4, 6, 8],

    "learning_rate": [0.03, 0.05, 0.10],

    "n_estimators": [200, 300],

    "subsample": [0.8],

    "colsample_bytree": [0.8],

    "min_child_weight": [1, 3],

    "gamma": [0, 0.2]
}


# ============================================================
# THRESHOLDS ⭐⭐⭐
# IMPORTANT:
# Higher because probabilities shifted upward
# ============================================================

thresholds = [

    0.40,
    0.50,
    0.60,
    0.70,
    0.80
]


# ============================================================
# STORE RESULTS
# ============================================================

results = []


# ============================================================
# HYPERPARAMETER + THRESHOLD SEARCH ⭐⭐⭐
# ============================================================

for params in ParameterGrid(param_grid):

    print(f"\nTesting Parameters: {params}")

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = XGBClassifier(

        random_state=42,

        eval_metric="logloss",

        scale_pos_weight=scale_pos_weight,

        **params
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.fit(

        X_train,

        y_train
    )

    # --------------------------------------------------------
    # PROBABILITIES
    # --------------------------------------------------------

    probabilities = model.predict_proba(

        X_test

    )[:,1]

    # --------------------------------------------------------
    # ROC AUC
    # --------------------------------------------------------

    auc = roc_auc_score(

        y_test,

        probabilities
    )

    # --------------------------------------------------------
    # TEST THRESHOLDS
    # --------------------------------------------------------

    for threshold in thresholds:

        predictions = (

            probabilities > threshold
        ).astype(int)

        precision = precision_score(

            y_test,

            predictions,

            zero_division=0
        )

        recall = recall_score(

            y_test,

            predictions
        )

        f1 = f1_score(

            y_test,

            predictions
        )

        results.append({

            "max_depth": params["max_depth"],

            "learning_rate": params["learning_rate"],

            "n_estimators": params["n_estimators"],

            "min_child_weight": params["min_child_weight"],

            "gamma": params["gamma"],

            "threshold": threshold,

            "roc_auc": auc,

            "precision": precision,

            "recall": recall,

            "f1_score": f1
        })

        print(

            f"Threshold={threshold} | "
            f"AUC={auc:.4f} | "
            f"Precision={precision:.4f} | "
            f"Recall={recall:.4f} | "
            f"F1={f1:.4f}"
        )

    print(
        "\n----------------------------------------"
    )


# ============================================================
# RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)


# ============================================================
# SORT RESULTS ⭐⭐⭐
# ============================================================

results_df = results_df.sort_values(

    by=[

        "f1_score",

        "roc_auc",

        "precision"
    ],

    ascending=False
)



(307511, 169)

scale_pos_weight = 11.39

Testing Parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}
Threshold=0.4 | AUC=0.7592 | Precision=0.1350 | Recall=0.8123 | F1=0.2315
Threshold=0.5 | AUC=0.7592 | Precision=0.1672 | Recall=0.6862 | F1=0.2689
Threshold=0.6 | AUC=0.7592 | Precision=0.2125 | Recall=0.5343 | F1=0.3041
Threshold=0.7 | AUC=0.7592 | Precision=0.2727 | Recall=0.3406 | F1=0.3029
Threshold=0.8 | AUC=0.7592 | Precision=0.4010 | Recall=0.1289 | F1=0.1951

----------------------------------------

Testing Parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
Threshold=0.4 | AUC=0.7620 | Precision=0.1369 | Recall=0.8062 | F1=0.2340
Threshold=0.5 | AUC=0.7620 | Precision=0.1693 | Recall=0.6886 | F1=0.2718
Threshold=0.6 | AUC=0.7620 | Precision=0.2137 | Recall=0.5416 | F1=0.3064
Thresh

In [60]:

# ============================================================
# TOP RESULTS
# ============================================================

print(
    "\nTOP 20 RESULTS\n"
)

print(

    results_df.head(20)
)


# ============================================================
# BEST MODEL CONFIG ⭐⭐⭐
# ============================================================

best_result = results_df.iloc[0]

print(
    "\nBEST CONFIGURATION\n"
)

print(best_result)


TOP 20 RESULTS

     max_depth  learning_rate  n_estimators  min_child_weight  gamma  \
292          8           0.05           200                 3    0.2   
128          4           0.10           300                 1    0.0   
308          4           0.10           300                 1    0.2   
297          8           0.05           300                 3    0.2   
123          4           0.10           200                 1    0.0   
303          4           0.10           200                 1    0.2   
98           6           0.05           300                 3    0.0   
268          6           0.05           300                 1    0.2   
87           6           0.05           300                 1    0.0   
277          6           0.05           300                 3    0.2   
88           6           0.05           300                 1    0.0   
97           6           0.05           300                 3    0.0   
92           6           0.05           200    

In [ ]:
# ============================================================
# ENTERPRISE CREDIT RISK PIPELINE ⭐⭐⭐
# SMOTEENN / SMOTETOMEK + XGBOOST
#
# Includes:
# - imbalance handling
# - hyperparameter optimization
# - threshold optimization
# - ROC-AUC
# - Precision
# - Recall
# - F1
# - confusion matrix
#
# ENTERPRISE-GRADE MODELING PIPELINE
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (

    train_test_split,

    ParameterGrid
)

from sklearn.metrics import (

    roc_auc_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix
)

from xgboost import XGBClassifier


# ============================================================
# LOAD DATASET
# ============================================================

application_train = pd.read_csv(

    "../data/processed/engineered_credit_risk_dataset.csv"
)

print(
    application_train.shape
)


# ============================================================
# FEATURE COLUMNS ⭐⭐⭐
# ============================================================

feature_columns = [

    # AFFORDABILITY

    "ANNUITY_TO_INCOME",

    "CREDIT_TO_INCOME",

    "FREE_CASH_FLOW",

    "DEBT_TO_INCOME",

    "CREDIT_TO_ANNUITY_RATIO",

    "CREDIT_TO_GOODS_RATIO",

    "DOWN_PAYMENT",

    # EXT_SOURCE

    "EXT_SOURCE_1",

    "EXT_SOURCE_2",

    "EXT_SOURCE_3",

    "EXT_SOURCE_MEAN",

    "EXT_SOURCE_STD",

    "CREDIT_EXT_RATIO",

    # LEVERAGE

    "DEBT_PER_BUREAU_RECORD",

    "OVERDUE_PER_BUREAU_RECORD",

    "ACTIVE_DEBT_RATIO",

    "MEAN_DAYS_CREDIT",

    "LAST_ACTIVE_DAYS_CREDIT",

    # BORROWING

    "APPLICATIONS_PER_INCOME",

    "PREVIOUS_APPLICATION_COUNT",

    "RECENT_APPLICATION_COUNT",

    "BORROWING_ACCELERATION_RATIO",

    # REPAYMENT

    "LATE_PAYMENT_COUNT",

    "MISSED_PAYMENTS_PER_LOAN",

    "AVG_PAYMENT_DELAY",

    "AVG_PAYMENT_DEFICIT",

    "REPAYMENT_STABILITY",

    # TEMPORAL

    "LATE_PAYMENTS_LAST_90D",

    "AVG_PAYMENT_DELAY_LAST_90D",

    "RECENT_TO_HISTORICAL_DELAY_RATIO",

    "RECENT_DELAY_TREND",

    "RECENT_PAYMENT_DEFICIT",

    "PAYMENT_DEFICIT_TREND",

    "RECENT_PAYMENT_STABILITY",

    # EMPLOYMENT

    "YEARS_EMPLOYED",

    "EMPLOYMENT_TO_AGE_RATIO",

    "AGE_YEARS",

    # HISTORICAL CAPACITY

    "MAX_INSTALLMENT",

    "ANNUITY_TO_MAX_INSTALLMENT_RATIO",

    # COMPOSITE

    "BEHAVIORAL_RISK_SCORE"
]


# ============================================================
# TARGET
# ============================================================

target_column = "TARGET"

X = application_train[
    feature_columns
]

y = application_train[
    target_column
]

X = X.fillna(0)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print(
    "\nOriginal Train Distribution\n"
)

print(
    y_train.value_counts()
)


# ============================================================
# CHOOSE RESAMPLING METHOD ⭐⭐⭐
# ============================================================

USE_SMOTEENN = True

USE_SMOTETOMEK = False


# ============================================================
# SMOTEENN ⭐⭐⭐
# ============================================================

if USE_SMOTEENN:

    from imblearn.combine import SMOTEENN

    print(
        "\nUsing SMOTEENN\n"
    )

    resampler = SMOTEENN(

        random_state=42
    )

    X_train_resampled, y_train_resampled = (

        resampler.fit_resample(

            X_train,

            y_train
        )
    )


# ============================================================
# SMOTETOMEK ⭐⭐⭐
# ============================================================

elif USE_SMOTETOMEK:

    from imblearn.combine import SMOTETomek

    print(
        "\nUsing SMOTETomek\n"
    )

    resampler = SMOTETomek(

        random_state=42
    )

    X_train_resampled, y_train_resampled = (

        resampler.fit_resample(

            X_train,

            y_train
        )
    )


# ============================================================
# RESAMPLED DISTRIBUTION
# ============================================================

print(
    "\nResampled Distribution\n"
)

print(
    y_train_resampled.value_counts()
)


# ============================================================
# XGBOOST PARAMETER GRID ⭐⭐⭐
# ============================================================

param_grid = {

    "max_depth": [4, 6, 8],

    "learning_rate": [0.03, 0.05, 0.10],

    "n_estimators": [200, 300],

    "subsample": [0.8],

    "colsample_bytree": [0.8],

    "min_child_weight": [1, 3],

    "gamma": [0, 0.2]
}


# ============================================================
# THRESHOLDS ⭐⭐⭐
# ============================================================

thresholds = [

    0.10,

    0.15,

    0.20,

    0.25,

    0.30,

    0.40,

    0.50
]


# ============================================================
# STORE RESULTS
# ============================================================

results = []


# ============================================================
# HYPERPARAMETER SEARCH ⭐⭐⭐
# ============================================================

for params in ParameterGrid(param_grid):

    print(
        f"\nTesting Parameters: {params}"
    )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = XGBClassifier(

        random_state=42,

        eval_metric="logloss",

        **params
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.fit(

        X_train_resampled,

        y_train_resampled
    )

    # --------------------------------------------------------
    # PREDICT PROBABILITIES
    # --------------------------------------------------------

    probabilities = model.predict_proba(

        X_test

    )[:,1]

    # --------------------------------------------------------
    # ROC AUC
    # --------------------------------------------------------

    auc = roc_auc_score(

        y_test,

        probabilities
    )

    # --------------------------------------------------------
    # THRESHOLD TESTING
    # --------------------------------------------------------

    for threshold in thresholds:

        predictions = (

            probabilities > threshold
        ).astype(int)

        precision = precision_score(

            y_test,

            predictions,

            zero_division=0
        )

        recall = recall_score(

            y_test,

            predictions
        )

        f1 = f1_score(

            y_test,

            predictions
        )

        results.append({

            "max_depth": params["max_depth"],

            "learning_rate": params["learning_rate"],

            "n_estimators": params["n_estimators"],

            "min_child_weight": params["min_child_weight"],

            "gamma": params["gamma"],

            "threshold": threshold,

            "roc_auc": auc,

            "precision": precision,

            "recall": recall,

            "f1_score": f1
        })

        print(

            f"Threshold={threshold} | "
            f"AUC={auc:.4f} | "
            f"Precision={precision:.4f} | "
            f"Recall={recall:.4f} | "
            f"F1={f1:.4f}"
        )

    print(
        "\n-------------------------------------------"
    )


# ============================================================
# RESULTS DATAFRAME ⭐⭐⭐
# ============================================================

results_df = pd.DataFrame(results)


# ============================================================
# SORT RESULTS
# ============================================================

results_df = results_df.sort_values(

    by=[

        "f1_score",

        "roc_auc",

        "precision"
    ],

    ascending=False
)


# ============================================================
# TOP RESULTS
# ============================================================

print(
    "\nTOP 20 RESULTS\n"
)

print(

    results_df.head(20)
)


# ============================================================
# BEST MODEL CONFIG ⭐⭐⭐
# ============================================================

best_result = results_df.iloc[0]

print(
    "\nBEST CONFIGURATION\n"
)

print(best_result)



(307511, 169)

Original Train Distribution

TARGET
0    226148
1     19860
Name: count, dtype: int64

Using SMOTEENN


Resampled Distribution

TARGET
1    203356
0    142298
Name: count, dtype: int64

Testing Parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}
Threshold=0.1 | AUC=0.7291 | Precision=0.0930 | Recall=0.9635 | F1=0.1696
Threshold=0.15 | AUC=0.7291 | Precision=0.1071 | Recall=0.8975 | F1=0.1914
Threshold=0.2 | AUC=0.7291 | Precision=0.1216 | Recall=0.8222 | F1=0.2119
Threshold=0.25 | AUC=0.7291 | Precision=0.1365 | Recall=0.7505 | F1=0.2310
Threshold=0.3 | AUC=0.7291 | Precision=0.1500 | Recall=0.6739 | F1=0.2454
Threshold=0.4 | AUC=0.7291 | Precision=0.1776 | Recall=0.5327 | F1=0.2664
Threshold=0.5 | AUC=0.7291 | Precision=0.2128 | Recall=0.3879 | F1=0.2748

-------------------------------------------

Testing Parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate':

In [64]:

# ============================================================
# RETRAIN BEST MODEL ⭐⭐⭐
# ============================================================

best_model = XGBClassifier(

    max_depth=int(best_result["max_depth"]),

    learning_rate=float(
        best_result["learning_rate"]
    ),

    n_estimators=int(
        best_result["n_estimators"]
    ),

    min_child_weight=int(
        best_result["min_child_weight"]
    ),

    gamma=float(
        best_result["gamma"]
    ),

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"
)

best_model.fit(

    X_train_resampled,

    y_train_resampled
)


# ============================================================
# FINAL PREDICTIONS ⭐⭐⭐
# ============================================================

best_threshold = float(
    best_result["threshold"]
)

final_probabilities = best_model.predict_proba(

    X_test

)[:,1]

final_predictions = (

    final_probabilities > best_threshold
).astype(int)


# ============================================================
# FINAL CLASSIFICATION REPORT
# ============================================================

print(
    "\nFINAL CLASSIFICATION REPORT\n"
)

print(

    classification_report(

        y_test,

        final_predictions
    )
)



FINAL CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       0.94      0.89      0.92     56538
           1       0.24      0.41      0.30      4965

    accuracy                           0.85     61503
   macro avg       0.59      0.65      0.61     61503
weighted avg       0.89      0.85      0.87     61503



In [65]:

# ============================================================
# FINAL ROC AUC
# ============================================================

final_auc = roc_auc_score(

    y_test,

    final_probabilities
)

print(
    f"\nFINAL ROC AUC: {final_auc:.4f}"
)


# ============================================================
# FINAL CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    final_predictions
)

print(
    "\nFINAL CONFUSION MATRIX\n"
)

print(cm)


FINAL ROC AUC: 0.7544

FINAL CONFUSION MATRIX

[[50218  6320]
 [ 2950  2015]]


In [66]:
# ============================================================
# SAVE FINAL PRODUCTION MODEL ⭐⭐⭐
# ============================================================

import joblib

# ------------------------------------------------------------
# SAVE BEST MODEL
# ------------------------------------------------------------

joblib.dump(

    best_model,

    "../models/final_credit_risk_xgboost_model.pkl"
)

print(
    "Final model saved successfully."
)


# ============================================================
# SAVE FEATURE LIST ⭐⭐⭐
# ============================================================

joblib.dump(

    feature_columns,

    "../models/final_feature_columns.pkl"
)

print(
    "Feature column list saved successfully."
)


# ============================================================
# SAVE FINAL THRESHOLD ⭐⭐⭐
# ============================================================

joblib.dump(

    best_threshold,

    "../models/final_threshold.pkl"
)

print(
    "Final threshold saved successfully."
)


# ============================================================
# SAVE MODEL METADATA ⭐⭐⭐
# ============================================================

model_metadata = {

    "model_type": "XGBoost",

    "resampling_strategy": "SMOTEENN",

    "roc_auc": 0.7544,

    "precision": 0.24,

    "recall": 0.41,

    "f1_score": 0.30,

    "threshold": best_threshold,

    "business_purpose": (

        "Credit risk ranking and "
        "early warning prioritization"
    )
}

joblib.dump(

    model_metadata,

    "../models/final_model_metadata.pkl"
)

print(
    "Model metadata saved successfully."
)

Final model saved successfully.
Feature column list saved successfully.
Final threshold saved successfully.
Model metadata saved successfully.
